In [2]:
import pandas as pd

print("=== TASK 1: Data Preparation ===\n")

# -------------------------------
# Create DataFrames
# -------------------------------
students_data = {
    'student_id': [101, 102, 103, 104, 105, 106, 107],
    'name': ['Alice', 'Bob', None, 'David', 'Emma', 'Frank', 'Grace'],
    'email': ['alice@email.com', 'bob@email.com', 'charlie@email.com', None, 'emma@email.com', 'frank@email.com', 'grace@email.com'],
    'city': ['Mumbai', 'Delhi', 'Bangalore', 'Mumbai', None, 'Chennai', 'Delhi']
}

enrollments_data = {
    'student_id': [101, 102, 103, 105, 108, 109],
    'course_name': ['Python', 'Data Science', 'Python', 'Machine Learning', 'AI', 'Python'],
    'enrollment_date': ['2024-01-15', '2024-01-20', '2024-02-01', '2024-02-10', '2024-02-15', '2024-03-01']
}

scores_data = {
    'student_id': [101, 102, 104, 105, 106],
    'exam_score': [85, 92, 78, 88, 95]
}

students = pd.DataFrame(students_data)
enrollments = pd.DataFrame(enrollments_data)
scores = pd.DataFrame(scores_data)

# -------------------------------
# Original Data
# -------------------------------
print("Original Students DataFrame:")
print(students, "\n")

# -------------------------------
# Null Analysis
# -------------------------------
print("Null Value Analysis:")
for col in students.columns:
    nulls = students[col].isnull().sum()
    percent = (nulls / len(students)) * 100
    print(f"Column: {col}, Nulls: {nulls} ({percent:.2f}%)")

# -------------------------------
# Cleaning
# -------------------------------
students['city'] = students['city'].fillna('Unknown')
students_cleaned = students.dropna(subset=['name'])

print("\nCleaned Students DataFrame ({} rows):".format(len(students_cleaned)))
print(students_cleaned)



# =====================================================
print("\n=== TASK 2: Join Operations ===\n")

# -------------------------------
# INNER JOIN
# -------------------------------
inner_join = pd.merge(students_cleaned, enrollments, on='student_id', how='inner')

print("Inner Join Result ({} rows):".format(len(inner_join)))
print(inner_join[['student_id', 'name', 'course_name']])

excluded = set(students_cleaned['student_id']) - set(enrollments['student_id'])
print("Excluded students:", list(excluded), "- Not in enrollments table\n")



# -------------------------------
# LEFT JOIN
# -------------------------------
left_join = pd.merge(students_cleaned, enrollments, on='student_id', how='left')

print("Left Join Result ({} rows):".format(len(left_join)))
print(left_join[['student_id', 'name', 'course_name']])

null_courses = left_join[left_join['course_name'].isnull()]['student_id'].tolist()
print("Students with null course_name:", null_courses, "\n")



# -------------------------------
# RIGHT JOIN
# -------------------------------
right_join = pd.merge(students_cleaned, enrollments, on='student_id', how='right')

print("Right Join Result ({} rows):".format(len(right_join)))
print(right_join[['student_id', 'name', 'course_name']])

missing_names = right_join[right_join['name'].isnull()]['student_id'].tolist()
print("Student IDs without names:", missing_names, "\n")




# -------------------------------
# OUTER JOIN
# -------------------------------
outer_join = pd.merge(students_cleaned, enrollments, on='student_id', how='outer')

print("Outer Join Result ({} rows):".format(len(outer_join)))
print(outer_join[['student_id', 'name', 'course_name']])

missing_rows = outer_join[
    outer_join['name'].isnull() | outer_join['course_name'].isnull()
]

print("\nRows with missing name OR course:")
print(missing_rows[['student_id', 'name', 'course_name']])



# Indicator
outer_indicator = pd.merge(
    students_cleaned,
    enrollments,
    on='student_id',
    how='outer',
    indicator=True
)

print("\nMerge Indicator Distribution:")
print(outer_indicator['_merge'].value_counts())

# =====================================================
print("\n=== TASK 3: Lookup and Automation ===\n")

# -------------------------------
# Lookup using map
# -------------------------------
score_dict = dict(zip(scores['student_id'], scores['exam_score']))
students_cleaned['exam_score'] = students_cleaned['student_id'].map(score_dict)

print("Lookup Operation Result:")
print(students_cleaned[['student_id', 'name', 'exam_score']])

# -------------------------------
# Merge alternative
# -------------------------------
students_scores_merge = pd.merge(
    students_cleaned,
    scores,
    on='student_id',
    how='left'
)

# Explanation:

#Why map() is faster than merge() for lookup:
#- map() performs a simple dictionary lookup → O(1) time for each row
#- merge() needs to:
    #• align rows
    #• sort/join tables
    #• manage extra columns
  #This adds overhead, making merge slower for single-column lookups.

# -------------------------------
# Automation Function
# -------------------------------
def auto_merge(df1, df2, join_type, key_column):
    merged_df = pd.merge(df1, df2, on=key_column, how=join_type)
    return {
        'result_df': merged_df,
        'row_count': len(merged_df),
        'join_type': join_type
    }

# -------------------------------
# Test Function
# -------------------------------
print("\nAutomation Function Test:\n")

result_inner = auto_merge(students_cleaned, enrollments, 'inner', 'student_id')
print("Join Type:", result_inner['join_type'])
print("Rows in Result:", result_inner['row_count'])
print(result_inner['result_df'][['student_id', 'name', 'course_name']], "\n")

result_left = auto_merge(students_cleaned, enrollments, 'left', 'student_id')
print("Join Type:", result_left['join_type'])
print("Rows in Result:", result_left['row_count'])
print(result_left['result_df'][['student_id', 'name', 'course_name']])

=== TASK 1: Data Preparation ===

Original Students DataFrame:
   student_id   name              email       city
0         101  Alice    alice@email.com     Mumbai
1         102    Bob      bob@email.com      Delhi
2         103   None  charlie@email.com  Bangalore
3         104  David               None     Mumbai
4         105   Emma     emma@email.com       None
5         106  Frank    frank@email.com    Chennai
6         107  Grace    grace@email.com      Delhi 

Null Value Analysis:
Column: student_id, Nulls: 0 (0.00%)
Column: name, Nulls: 1 (14.29%)
Column: email, Nulls: 1 (14.29%)
Column: city, Nulls: 1 (14.29%)

Cleaned Students DataFrame (6 rows):
   student_id   name            email     city
0         101  Alice  alice@email.com   Mumbai
1         102    Bob    bob@email.com    Delhi
3         104  David             None   Mumbai
4         105   Emma   emma@email.com  Unknown
5         106  Frank  frank@email.com  Chennai
6         107  Grace  grace@email.com    Delhi

=== 

C:\Users\Asus\AppData\Local\Temp\ipykernel_1092\1180935500.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  students_cleaned['exam_score'] = students_cleaned['student_id'].map(score_dict)
